# 01 — Feature Engineering

## ML Perovskites Project

This notebook transforms the audited raw dataset into machine-learning-ready
feature matrices based on chemical composition and elemental properties.

### Input tables

- `ml_perovskites.db_ml`
- `ml_perovskites.atom_props`

### Main objectives

1. Parse chemical compositions.
2. Identify the elements present in each compound.
3. Retrieve elemental properties from `atom_props`.
4. Generate composition-weighted elemental descriptors.
5. Generate intrinsic structural descriptors where applicable.
6. Preserve experimental variables separately.
7. Build target-specific feature matrices.

### Target models

#### Bandgap
Uses intrinsic/material descriptors only.

#### Hydrogen production
May use intrinsic/material descriptors together with experimentally
justifiable reaction/material variables.

### Important modeling rule

Target-derived information must never be used as an input feature.

No machine-learning model is trained in this notebook.

In [0]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DB_ML_TABLE = "ml_perovskites.db_ml"
ATOM_PROPS_TABLE = "ml_perovskites.atom_props"

COMPOUND_CANDIDATES = [
    "Compound",
    "compound"
]

BANDGAP_CANDIDATES = [
    "bandgap_energy"
]

HYDROGEN_CANDIDATES = [
    "hydrogen_production_rate"
]

print("db_ml table:", DB_ML_TABLE)
print("atom_props table:", ATOM_PROPS_TABLE)

In [0]:
# ============================================================
# 2. IMPORTS
# ============================================================

import re
import math
import numpy as np
import pandas as pd

from pyspark.sql import functions as F

print("Imports loaded.")

In [0]:
# ============================================================
# 3. LOAD SOURCE TABLES
# ============================================================

db_ml = spark.table(DB_ML_TABLE).toPandas()

# KEEP atom_props in Spark
atom_props_spark = spark.table(ATOM_PROPS_TABLE)

print(
    f"db_ml shape: {db_ml.shape}"
)

print(
    f"atom_props shape: "
    f"({atom_props_spark.count():,}, "
    f"{len(atom_props_spark.columns)})"
)

In [0]:
# ============================================================
# 4. IDENTIFY KEY COLUMNS
# ============================================================

def find_col(df, candidates):

    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        if candidate.lower() in lookup:
            return lookup[candidate.lower()]

    return None


compound_col = find_col(
    db_ml,
    COMPOUND_CANDIDATES
)

bandgap_col = find_col(
    db_ml,
    BANDGAP_CANDIDATES
)

h2_col = find_col(
    db_ml,
    HYDROGEN_CANDIDATES
)

element_col = find_col(
    atom_props_spark,
    ["Element", "element"]
)

print("Compound column:", compound_col)
print("Bandgap column:", bandgap_col)
print("Hydrogen column:", h2_col)
print("Element column:", element_col)

In [0]:
# ============================================================
# 5. VALIDATE KEY COLUMNS
# ============================================================

required_columns = {
    "compound": compound_col,
    "bandgap": bandgap_col,
    "hydrogen": h2_col,
    "element": element_col
}

missing_required = [
    name
    for name, col in required_columns.items()
    if col is None
]

if missing_required:

    raise ValueError(
        "Required columns not found: "
        + ", ".join(missing_required)
    )

print("All required columns found.")

In [0]:
# ============================================================
# 6. COMPOSITION INVENTORY
# ============================================================

composition_inventory = (
    db_ml[[compound_col]]
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
)

print(
    "Unique non-null compositions:",
    len(composition_inventory)
)

display(
    composition_inventory.head(30)
)

In [0]:
# ============================================================
# 7. CHEMICAL FORMULA PARSER
# ============================================================

ELEMENT_PATTERN = re.compile(
    r"([A-Z][a-z]?)([0-9]*\.?[0-9]*)"
)


def parse_formula(formula):

    if pd.isna(formula):
        return {}

    formula = str(formula).strip()

    if not formula:
        return {}

    matches = ELEMENT_PATTERN.findall(formula)

    composition = {}

    for element, amount in matches:

        if not element:
            continue

        if amount == "":
            amount = 1.0
        else:
            amount = float(amount)

        composition[element] = (
            composition.get(element, 0.0)
            + amount
        )

    return composition


test_formulas = [
    "TiO2",
    "SrTiO3",
    "CsPbBr3",
    "FA0.8MA0.2PbI3"
]

for formula in test_formulas:

    print(
        formula,
        "->",
        parse_formula(formula)
    )

In [0]:
# ============================================================
# 8. PARSER TEST
# ============================================================

test_formulas = [
    "TiO2",
    "SrTiO3",
    "CsPbBr3"
]

for formula in test_formulas:

    print(
        formula,
        "->",
        parse_formula(formula)
    )

In [0]:
# ============================================================
# 8B. CONSERVATIVE CHEMICAL FORMULA PARSER
# ============================================================

import re

# Valid chemical element symbols
VALID_ELEMENTS = {
    "H", "He", "Li", "Be", "B", "C", "N", "O", "F", "Ne",
    "Na", "Mg", "Al", "Si", "P", "S", "Cl", "Ar", "K", "Ca",
    "Sc", "Ti", "V", "Cr", "Mn", "Fe", "Co", "Ni", "Cu", "Zn",
    "Ga", "Ge", "As", "Se", "Br", "Kr", "Rb", "Sr", "Y", "Zr",
    "Nb", "Mo", "Tc", "Ru", "Rh", "Pd", "Ag", "Cd", "In", "Sn",
    "Sb", "Te", "I", "Xe", "Cs", "Ba", "La", "Ce", "Pr", "Nd",
    "Pm", "Sm", "Eu", "Gd", "Tb", "Dy", "Ho", "Er", "Tm", "Yb",
    "Lu", "Hf", "Ta", "W", "Re", "Os", "Ir", "Pt", "Au", "Hg",
    "Tl", "Pb", "Bi", "Po", "At", "Rn", "Fr", "Ra", "Ac", "Th",
    "Pa", "U", "Np", "Pu", "Am", "Cm", "Bk", "Cf", "Es", "Fm",
    "Md", "No", "Lr", "Rf", "Db", "Sg", "Bh", "Hs", "Mt", "Ds",
    "Rg", "Cn", "Nh", "Fl", "Mc", "Lv", "Ts", "Og"
}

ELEMENT_REGEX = re.compile(
    r"([A-Z][a-z]?)([0-9]*\.?[0-9]*)"
)

VARIABLE_PATTERN = re.compile(
    r"[xyzXYZ]"
)


def parse_formula_conservative(formula):

    if pd.isna(formula):
        return {}, "missing"

    formula = str(formula).strip()

    if not formula:
        return {}, "missing"

    # Variable stoichiometry
    if VARIABLE_PATTERN.search(formula):
        return {}, "variable_stoichiometry"

    # Special notation such as TBA/...
    if "/" in formula:
        return {}, "special_notation"

    # Known non-standard notation
    if any(
        token.lower() in formula.lower()
        for token in ["TBA", "SAC", "Gb"]
    ):
        return {}, "special_notation"

    matches = ELEMENT_REGEX.findall(formula)

    if not matches:
        return {}, "unparsed"

    composition = {}

    for element, amount in matches:

        if element not in VALID_ELEMENTS:
            return {}, "invalid_element"

        if amount == "":
            amount = 1.0
        else:
            amount = float(amount)

        composition[element] = (
            composition.get(element, 0.0)
            + amount
        )

    return composition, "valid"


print("Conservative formula parser loaded successfully.")

In [0]:
# ============================================================
# 9. APPLY CONSERVATIVE PARSER
# ============================================================

parsed_results = (
    db_ml[compound_col]
    .apply(parse_formula_conservative)
)

db_ml["parsed_composition"] = (
    parsed_results
    .apply(lambda x: x[0])
)

db_ml["parse_status"] = (
    parsed_results
    .apply(lambda x: x[1])
)

db_ml["n_elements"] = (
    db_ml["parsed_composition"]
    .apply(len)
)

print("PARSING SUMMARY")
print("-" * 50)

print(
    db_ml["parse_status"]
    .value_counts(dropna=False)
)

print()

print(
    "Valid formulas:",
    int(
        (
            db_ml["parse_status"] == "valid"
        ).sum()
    )
)

print(
    "Variable stoichiometry:",
    int(
        (
            db_ml["parse_status"]
            == "variable_stoichiometry"
        ).sum()
    )
)

print(
    "Special notation:",
    int(
        (
            db_ml["parse_status"]
            == "special_notation"
        ).sum()
    )
)

print(
    "Other parsing problems:",
    int(
        ~db_ml["parse_status"].isin(
            [
                "valid",
                "variable_stoichiometry",
                "special_notation"
            ]
        ).sum()
    )
)

In [0]:
# ============================================================
# 9B. VALIDATE PARSED ELEMENTS
# ============================================================

invalid_elements = set()

for composition in db_ml["parsed_composition"]:

    for element in composition.keys():

        if element not in VALID_ELEMENTS:
            invalid_elements.add(element)

print(
    "Invalid/non-element tokens:",
    len(invalid_elements)
)

print(
    sorted(invalid_elements)
)

In [0]:
# ============================================================
# 9C. NON-STANDARD FORMULA AUDIT
# ============================================================

nonstandard_formulas = db_ml[
    db_ml["parse_status"] != "valid"
][
    [
        compound_col,
        "parse_status"
    ]
].drop_duplicates()

print(
    "Non-standard unique formulas:",
    len(nonstandard_formulas)
)

display(nonstandard_formulas)

In [0]:
# ============================================================
# 9D. VALID ELEMENT COVERAGE
# ============================================================

valid_compositions = db_ml[
    db_ml["parse_status"] == "valid"
]["parsed_composition"]

valid_observed_elements = set()

for composition in valid_compositions:
    valid_observed_elements.update(
        composition.keys()
    )

atom_props_elements = set(
    row[element_col]
    for row in (
        atom_props_spark
        .select(element_col)
        .where(
            F.col(element_col).isNotNull()
        )
        .distinct()
        .collect()
    )
)

missing_valid_elements = sorted(
    valid_observed_elements
    - atom_props_elements
)

print(
    "Elements in valid formulas:",
    len(valid_observed_elements)
)

print(
    "Covered by atom_props:",
    len(
        valid_observed_elements
        & atom_props_elements
    )
)

print(
    "Missing from atom_props:",
    len(missing_valid_elements)
)

if missing_valid_elements:
    print(
        "Missing valid elements:",
        missing_valid_elements
    )
else:
    print(
        "All elements from valid formulas "
        "are covered by atom_props."
    )

In [0]:
# ============================================================
# 10. NORMALIZE VALID COMPOSITIONS
# ============================================================

def normalize_composition(composition):

    if not composition:
        return {}

    total = sum(composition.values())

    if total <= 0:
        return {}

    return {
        element: amount / total
        for element, amount in composition.items()
    }


db_ml["normalized_composition"] = (
    db_ml["parsed_composition"]
    .apply(normalize_composition)
)

print(
    "Normalized valid compositions:",
    int(
        (
            db_ml["parse_status"] == "valid"
        ).sum()
    )
)

display(
    db_ml[
        [
            compound_col,
            "parse_status",
            "normalized_composition"
        ]
    ].head(20)
)

In [0]:
# ============================================================
# 11. LOAD ELEMENTAL PROPERTIES
# ============================================================

atom_props_pd = (
    atom_props_spark
    .toPandas()
)

print(
    "atom_props shape:",
    atom_props_pd.shape
)

print(
    "Element column:",
    element_col
)

display(
    atom_props_pd.head()
)

In [0]:
# ============================================================
# 12. CLEAN ELEMENTAL PROPERTY TABLE
# ============================================================

atom_props_pd[element_col] = (
    atom_props_pd[element_col]
    .astype(str)
    .str.strip()
)

atom_props_pd = (
    atom_props_pd
    .drop_duplicates(
        subset=[element_col]
    )
    .reset_index(drop=True)
)

print(
    "Unique elements:",
    atom_props_pd[element_col].nunique()
)

In [0]:
# ============================================================
# 13. ELEMENT PROPERTY LOOKUP
# ============================================================

element_property_lookup = (
    atom_props_pd
    .set_index(element_col)
    .to_dict(orient="index")
)

print(
    "Elements available:",
    len(element_property_lookup)
)

In [0]:
# ============================================================
# 14. NUMERIC ELEMENTAL PROPERTIES
# ============================================================

element_numeric_cols = []

for c in atom_props_pd.columns:

    if c == element_col:
        continue

    numeric_version = pd.to_numeric(
        atom_props_pd[c],
        errors="coerce"
    )

    if numeric_version.notna().sum() > 0:
        element_numeric_cols.append(c)

print(
    "Numeric elemental properties:",
    len(element_numeric_cols)
)

for c in element_numeric_cols:
    print(c)

In [0]:
# ============================================================
# 15. COMPOSITION-WEIGHTED DESCRIPTOR FUNCTION
# ============================================================

def weighted_descriptor(
    composition,
    property_name
):

    if not composition:
        return np.nan

    values = []
    weights = []

    for element, fraction in composition.items():

        if element not in element_property_lookup:
            continue

        raw_value = (
            element_property_lookup[element]
            .get(property_name)
        )

        value = pd.to_numeric(
            raw_value,
            errors="coerce"
        )

        if pd.isna(value):
            continue

        values.append(float(value))
        weights.append(float(fraction))

    if not values:
        return np.nan

    values = np.array(values)
    weights = np.array(weights)

    weight_sum = weights.sum()

    if weight_sum <= 0:
        return np.nan

    return np.sum(
        values * weights
    ) / weight_sum

In [0]:
# ============================================================
# 16. GENERATE COMPOSITION-WEIGHTED DESCRIPTORS
# ============================================================

descriptor_data = {}

for property_name in element_numeric_cols:

    feature_name = (
        "mean_"
        + re.sub(
            r"[^a-zA-Z0-9]+",
            "_",
            str(property_name).strip().lower()
        ).strip("_")
    )

    descriptor_data[feature_name] = (
        db_ml["normalized_composition"]
        .apply(
            lambda comp:
            weighted_descriptor(
                comp,
                property_name
            )
        )
    )

descriptor_df = pd.DataFrame(
    descriptor_data
)

print(
    "Descriptor matrix:",
    descriptor_df.shape
)

display(
    descriptor_df.head()
)

In [0]:
# ============================================================
# 17. DESCRIPTOR MISSINGNESS
# ============================================================

descriptor_missingness = (
    descriptor_df
    .isna()
    .sum()
    .to_frame("missing")
)

descriptor_missingness["missing_pct"] = (
    100
    * descriptor_missingness["missing"]
    / len(descriptor_df)
)

descriptor_missingness = (
    descriptor_missingness
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

display(
    descriptor_missingness
)

In [0]:
# ============================================================
# 18. DESCRIPTOR COVERAGE AUDIT
# ============================================================

descriptor_complete_mask = (
    descriptor_df.notna().all(axis=1)
)

print("=" * 60)
print("DESCRIPTOR COVERAGE")
print("=" * 60)

print(
    "Total rows:",
    len(descriptor_df)
)

print(
    "Rows with complete descriptors:",
    int(descriptor_complete_mask.sum())
)

print(
    "Rows with incomplete descriptors:",
    int((~descriptor_complete_mask).sum())
)

print(
    "Complete coverage (%):",
    round(
        100 * descriptor_complete_mask.mean(),
        2
    )
)

print()

print(
    "Expected valid formulas:",
    int(
        (
            db_ml["parse_status"]
            == "valid"
        ).sum()
    )
)

In [0]:
# ============================================================
# 19. VERIFY DESCRIPTOR MISSINGNESS
# ============================================================

verification = pd.DataFrame({
    "parse_status": db_ml["parse_status"],
    "descriptor_complete": descriptor_complete_mask
})

verification_summary = (
    pd.crosstab(
        verification["parse_status"],
        verification["descriptor_complete"]
    )
)

display(verification_summary)

In [0]:
# ============================================================
# 20. BUILD MASTER FEATURE DATASET
# ============================================================

feature_engineered = pd.concat(
    [
        db_ml.reset_index(drop=True),
        descriptor_df.reset_index(drop=True)
    ],
    axis=1
)

print(
    "Master feature dataset:",
    feature_engineered.shape
)

print(
    "Rows:",
    len(feature_engineered)
)

print(
    "Columns:",
    len(feature_engineered.columns)
)

In [0]:
# ============================================================
# 21. COMPOSITION-VALID SUBSET
# ============================================================

composition_valid_mask = (
    db_ml["parse_status"] == "valid"
)

composition_valid_df = (
    feature_engineered.loc[
        composition_valid_mask
    ]
    .reset_index(drop=True)
)

print(
    "Composition-valid rows:",
    len(composition_valid_df)
)

print(
    "Expected:",
    389
)

print(
    "Composition-valid dataset:",
    composition_valid_df.shape
)

In [0]:
# ============================================================
# 22. TARGET COVERAGE
# ============================================================

bandgap_valid = pd.to_numeric(
    composition_valid_df[bandgap_col],
    errors="coerce"
)

hydrogen_valid = pd.to_numeric(
    composition_valid_df[h2_col],
    errors="coerce"
)

target_coverage = pd.DataFrame({
    "target": [
        "bandgap_energy",
        "hydrogen_production_rate"
    ],
    "total_valid_composition_rows": [
        len(composition_valid_df),
        len(composition_valid_df)
    ],
    "valid_target_values": [
        int(bandgap_valid.notna().sum()),
        int(hydrogen_valid.notna().sum())
    ],
    "missing_target_values": [
        int(bandgap_valid.isna().sum()),
        int(hydrogen_valid.isna().sum())
    ]
})

display(target_coverage)

In [0]:
# ============================================================
# 23. DEFINE COMPOSITION DESCRIPTORS
# ============================================================

descriptor_columns = list(
    descriptor_df.columns
)

print(
    "Number of descriptors:",
    len(descriptor_columns)
)

print(
    descriptor_columns
)

In [0]:
# ============================================================
# 24. BANDGAP FEATURE MATRIX
# ============================================================

bandgap_dataset = composition_valid_df[
    descriptor_columns
].copy()

bandgap_dataset["bandgap_target"] = (
    pd.to_numeric(
        composition_valid_df[bandgap_col],
        errors="coerce"
    )
)

bandgap_dataset = bandgap_dataset[
    bandgap_dataset["bandgap_target"].notna()
].reset_index(drop=True)

print(
    "Bandgap modeling dataset:",
    bandgap_dataset.shape
)

print(
    "Bandgap samples:",
    len(bandgap_dataset)
)

display(
    bandgap_dataset.head()
)

In [0]:
# ============================================================
# 25. HYDROGEN FEATURE MATRIX
# ============================================================

hydrogen_dataset = composition_valid_df[
    descriptor_columns
].copy()

hydrogen_dataset["hydrogen_target"] = (
    pd.to_numeric(
        composition_valid_df[h2_col],
        errors="coerce"
    )
)

hydrogen_dataset = hydrogen_dataset[
    hydrogen_dataset["hydrogen_target"].notna()
].reset_index(drop=True)

print(
    "Hydrogen modeling dataset:",
    hydrogen_dataset.shape
)

print(
    "Hydrogen samples:",
    len(hydrogen_dataset)
)

display(
    hydrogen_dataset.head()
)

In [0]:
# ============================================================
# 26. DESCRIPTOR QUALITY AUDIT — CONSTANT FEATURES
# ============================================================

X_descriptors = descriptor_df.copy()

constant_features = [
    c
    for c in X_descriptors.columns
    if X_descriptors[c].nunique(
        dropna=True
    ) <= 1
]

print(
    "Total descriptors:",
    X_descriptors.shape[1]
)

print(
    "Constant descriptors:",
    len(constant_features)
)

if constant_features:

    print(
        "Constant features:"
    )

    for c in constant_features:
        print(" -", c)
else:

    print(
        "No constant descriptors found."
    )

In [0]:
# ============================================================
# 27. NEAR-CONSTANT FEATURE AUDIT
# ============================================================

near_constant_rows = []

for c in X_descriptors.columns:

    x = X_descriptors[c].dropna()

    if len(x) == 0:
        continue

    dominant_pct = (
        100
        * x.value_counts().iloc[0]
        / len(x)
    )

    if dominant_pct >= 95:

        near_constant_rows.append({
            "feature": c,
            "dominant_value_pct": dominant_pct,
            "n_unique": x.nunique()
        })

near_constant_df = pd.DataFrame(
    near_constant_rows
)

if len(near_constant_df) > 0:

    display(
        near_constant_df.sort_values(
            "dominant_value_pct",
            ascending=False
        )
    )

else:

    print(
        "No near-constant descriptors "
        "with >=95% identical values."
    )

In [0]:
# ============================================================
# 28. EXACT DUPLICATE FEATURE AUDIT
# ============================================================

duplicate_features = []

columns = list(X_descriptors.columns)

for i in range(len(columns)):

    for j in range(i + 1, len(columns)):

        c1 = columns[i]
        c2 = columns[j]

        if X_descriptors[c1].equals(
            X_descriptors[c2]
        ):

            duplicate_features.append({
                "feature_1": c1,
                "feature_2": c2
            })

duplicate_feature_df = pd.DataFrame(
    duplicate_features
)

print(
    "Exact duplicate feature pairs:",
    len(duplicate_feature_df)
)

if len(duplicate_feature_df) > 0:
    display(duplicate_feature_df)

In [0]:
# ============================================================
# 29. DESCRIPTOR-DESCRIPTOR CORRELATION
# ============================================================

correlation_matrix = (
    X_descriptors
    .corr(
        method="pearson",
        min_periods=100
    )
)

print(
    "Correlation matrix shape:",
    correlation_matrix.shape
)

display(
    correlation_matrix
)

In [0]:
# ============================================================
# 30. HIGH-CORRELATION FEATURE PAIRS
# ============================================================

CORRELATION_THRESHOLD = 0.95

high_corr_pairs = []

corr_columns = list(
    correlation_matrix.columns
)

for i in range(len(corr_columns)):

    for j in range(i + 1, len(corr_columns)):

        c1 = corr_columns[i]
        c2 = corr_columns[j]

        r = correlation_matrix.loc[
            c1,
            c2
        ]

        if pd.notna(r) and abs(r) >= CORRELATION_THRESHOLD:

            high_corr_pairs.append({
                "feature_1": c1,
                "feature_2": c2,
                "pearson_r": r,
                "abs_r": abs(r)
            })

high_corr_df = pd.DataFrame(
    high_corr_pairs
).sort_values(
    "abs_r",
    ascending=False
)

print(
    "Highly correlated pairs:",
    len(high_corr_df)
)

display(
    high_corr_df
)

In [0]:
# ============================================================
# 31. FEATURE REDUCTION — INITIAL CANDIDATES
# ============================================================

# Remove near-constant descriptors
near_constant_features = (
    near_constant_df["feature"]
    .tolist()
    if len(near_constant_df) > 0
    else []
)

# Remove exact duplicates
exact_duplicate_features = set()

if len(duplicate_feature_df) > 0:

    for _, row in duplicate_feature_df.iterrows():

        exact_duplicate_features.add(
            row["feature_2"]
        )

features_after_basic_filter = [
    c
    for c in descriptor_columns
    if c not in near_constant_features
    and c not in exact_duplicate_features
]

print(
    "Original descriptors:",
    len(descriptor_columns)
)

print(
    "Near-constant removed:",
    len(near_constant_features)
)

print(
    "Exact duplicates removed:",
    len(exact_duplicate_features)
)

print(
    "Descriptors remaining:",
    len(features_after_basic_filter)
)

In [0]:
# ============================================================
# 32. SCIENTIFIC CORRELATION REDUCTION
# ============================================================

corr_threshold = 0.95

corr_matrix_filtered = (
    X_descriptors[
        features_after_basic_filter
    ]
    .corr(
        method="pearson",
        min_periods=100
    )
)

# ------------------------------------------------------------
# Descriptor priority
#
# Higher priority = preferred when two descriptors are
# strongly correlated.
# ------------------------------------------------------------

descriptor_priority = {

    # Basic atomic properties
    "mean_atomic_number": 100,
    "mean_atomic_weight": 90,

    # Electronegativity
    "mean_electron_negativity": 100,
    "mean_en_pauling": 90,
    "mean_en_allen": 80,
    "mean_en_ghosh": 70,

    # Atomic radius
    "mean_atomic_radius": 100,
    "mean_atomic_radius_rahm": 90,

    # Covalent radius
    "mean_covalent_radius_cordero": 100,
    "mean_covalent_radius_pyykko": 90,
    "mean_covalent_radius_pyykko_double": 80,
    "mean_covalent_radius_pyykko_triple": 70,
    "mean_covalent_radius_slater": 60,

    # van der Waals radius
    "mean_vdw_radius": 100,
    "mean_vdw_radius_alvarez": 90,
    "mean_vdw_radius_mm3": 80,
    "mean_vdw_radius_uff": 70,

    # Polarizability
    "mean_dipole_polarizability": 100,
    "mean_polarizability": 90,

    # Electronic configuration
    "mean_num_s_unfilled": 100,
    "mean_num_s_valence": 90,

    "mean_num_p_unfilled": 100,
    "mean_num_p_valence": 90,

    "mean_num_d_unfilled": 100,
    "mean_num_d_valence": 90,

    "mean_num_f_unfilled": 100,
    "mean_num_f_valence": 90,

    # Crystal estimates
    "mean_gs_est_bcc_latcnt": 100,
    "mean_gs_est_fcc_latcnt": 90
}


features_to_remove_corr = set()
correlation_reduction_rows = []

features_available = (
    features_after_basic_filter.copy()
)

for i in range(
    len(features_available)
):

    for j in range(
        i + 1,
        len(features_available)
    ):

        f1 = features_available[i]
        f2 = features_available[j]

        if (
            f1 in features_to_remove_corr
            or f2 in features_to_remove_corr
        ):
            continue

        r = corr_matrix_filtered.loc[
            f1,
            f2
        ]

        if pd.isna(r):
            continue

        if abs(r) >= corr_threshold:

            p1 = descriptor_priority.get(
                f1,
                50
            )

            p2 = descriptor_priority.get(
                f2,
                50
            )

            # Keep the higher-priority descriptor
            if p1 >= p2:

                keep = f1
                remove = f2

            else:

                keep = f2
                remove = f1

            features_to_remove_corr.add(
                remove
            )

            correlation_reduction_rows.append({
                "feature_kept": keep,
                "feature_removed": remove,
                "pearson_r": r,
                "abs_r": abs(r),
                "priority_kept": max(p1, p2),
                "priority_removed": min(p1, p2)
            })


features_after_correlation = [
    c
    for c in features_after_basic_filter
    if c not in features_to_remove_corr
]

print(
    "Correlation threshold:",
    corr_threshold
)

print(
    "Features removed:",
    len(features_to_remove_corr)
)

print(
    "Features remaining:",
    len(features_after_correlation)
)

In [0]:
# ============================================================
# 33. FEATURE REDUCTION AUDIT
# ============================================================

feature_reduction_rows = []

# Near-constant features
for feature in near_constant_features:

    feature_reduction_rows.append({
        "feature": feature,
        "reason": "near_constant",
        "kept_representative": None,
        "pearson_r": None
    })


# Correlated features
for row in correlation_reduction_rows:

    feature_reduction_rows.append({
        "feature": row["feature_removed"],
        "reason": "high_correlation",
        "kept_representative": row[
            "feature_kept"
        ],
        "pearson_r": row[
            "pearson_r"
        ]
    })


feature_reduction_audit = pd.DataFrame(
    feature_reduction_rows
)

print(
    "Total features flagged for removal:",
    len(feature_reduction_audit)
)

display(
    feature_reduction_audit
)

In [0]:
# ============================================================
# 34. FINAL DESCRIPTOR CANDIDATES
# ============================================================

final_descriptor_candidates = (
    features_after_correlation.copy()
)

print(
    "Original descriptors:",
    len(descriptor_columns)
)

print(
    "Near-constant removed:",
    len(near_constant_features)
)

print(
    "Correlation-redundant removed:",
    len(features_to_remove_corr)
)

print(
    "Final descriptor candidates:",
    len(final_descriptor_candidates)
)

print()
print("FINAL DESCRIPTORS")
print("-" * 60)

for feature in final_descriptor_candidates:
    print(feature)

In [0]:
# ============================================================
# 35. FINAL DESCRIPTOR SANITY CHECK
# ============================================================

assert (
    len(final_descriptor_candidates)
    > 0
)

assert (
    len(
        set(final_descriptor_candidates)
    )
    == len(final_descriptor_candidates)
)

assert not any(
    c in near_constant_features
    for c in final_descriptor_candidates
)

print(
    "Final descriptor set passed sanity checks."
)

print(
    "Number of descriptors:",
    len(final_descriptor_candidates)
)

In [0]:
# ============================================================
# 36. REDUCED FEATURE MATRIX
# ============================================================

X_reduced = X_descriptors[
    final_descriptor_candidates
].copy()

print(
    "Reduced feature matrix:",
    X_reduced.shape
)

print(
    "Number of features:",
    X_reduced.shape[1]
)

print(
    "Number of rows:",
    X_reduced.shape[0]
)

In [0]:
# ============================================================
# 37. REDUCED FEATURE MISSINGNESS
# ============================================================

reduced_missingness = (
    X_reduced
    .isna()
    .sum()
    .to_frame("missing")
)

reduced_missingness["missing_pct"] = (
    100
    * reduced_missingness["missing"]
    / len(X_reduced)
)

reduced_missingness = (
    reduced_missingness
    .sort_values(
        "missing_pct",
        ascending=False
    )
)

display(
    reduced_missingness
)

In [0]:
# ============================================================
# 38. FEATURE DISTRIBUTION AUDIT
# ============================================================

distribution_rows = []

for c in X_reduced.columns:

    x = X_reduced[c].dropna()

    distribution_rows.append({
        "feature": c,
        "n": len(x),
        "min": x.min(),
        "q01": x.quantile(0.01),
        "q25": x.quantile(0.25),
        "median": x.median(),
        "q75": x.quantile(0.75),
        "q99": x.quantile(0.99),
        "max": x.max(),
        "mean": x.mean(),
        "std": x.std(),
        "skewness": x.skew()
    })

distribution_audit = pd.DataFrame(
    distribution_rows
)

display(
    distribution_audit
)

In [0]:
# ============================================================
# 39. SKEWNESS AUDIT
# ============================================================

SKEW_THRESHOLD = 2.0

strongly_skewed = (
    distribution_audit[
        distribution_audit["skewness"].abs()
        >= SKEW_THRESHOLD
    ]
    .sort_values(
        "skewness",
        key=lambda x: x.abs(),
        ascending=False
    )
)

print(
    "Features with |skewness| >= 2:",
    len(strongly_skewed)
)

display(
    strongly_skewed
)

In [0]:
# ============================================================
# 40. OUTLIER AUDIT — IQR
# ============================================================

outlier_rows = []

for c in X_reduced.columns:

    x = X_reduced[c].dropna()

    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)

    iqr = q3 - q1

    if iqr == 0:
        outlier_count = 0
    else:

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_count = int(
            (
                (x < lower)
                |
                (x > upper)
            ).sum()
        )

    outlier_rows.append({
        "feature": c,
        "outlier_count": outlier_count,
        "outlier_pct":
            100 * outlier_count / len(x)
    })

outlier_audit = pd.DataFrame(
    outlier_rows
).sort_values(
    "outlier_pct",
    ascending=False
)

display(
    outlier_audit
)